In [25]:
import json

def load_json(filename):
    with open(filename) as file:
        return json.load(file)

In [ ]:
import pandas as pd

filename = "6,7,8"
output_path = f"{filename}.pdf"
report_path = f"../../outputs/reports/2025/{filename}.json"
text_segmentation_path = f"../../outputs/text_segmentation/2025/{filename}.json"

report_df = pd.read_json(report_path)
report = load_json(report_path)
text_segmentation = load_json(text_segmentation_path)

In [27]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def reviews_date_plot(df: pd.DataFrame, method_name: str, pdf: PdfPages):
    reviews_per_day = df.groupby(df['timestamp'].dt.date).size()
    
    fig, ax = plt.subplots(figsize=(10, 10))
    reviews_per_day.plot(kind='bar', ax=ax)
    ax.set_title(f"Reviews Plot by Date with {method_name} Method")
    ax.set_xlabel("Date")
    ax.set_ylabel("Review Count")
    plt.xticks(rotation=45)
    
    pdf.savefig(fig)
    plt.close(fig)

In [28]:
with PdfPages(output_path) as pdf:
    for report_val in report.values():
        reviews = []
        for idx in report_val['summary_id']:
            selected_review = text_segmentation.get(f'{idx}')
            reviews.append({
                'id': idx,
                'text': selected_review['text'],
                'timestamp': selected_review['timestamp'],
            })

        reviews_df = pd.DataFrame(reviews)
        reviews_df['timestamp'] = pd.to_datetime(reviews_df['timestamp'])

        reviews_date_plot(reviews_df, report_val['decay_method'], pdf)